# Image Editing

In [ ]:
import os
import sys
import json
import random
import visdom
import logging
import textwrap
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm
from datetime import datetime


from matplotlib import pyplot as plt
# from matplotlib.ticker import PercentFormatter
import seaborn as sns
sns.set_style("dark")

import tomllib
import avoddiag as ag

## Load Configuration

In [ ]:
with open("config/config.toml", "rb") as f:
    config = tomllib.load(f)

## Load Image Dataset

In [ ]:
image_dataset = ag.image.data.Dataset(
    root_folder_path="output/TestDataset_01_Generated_Images"
)

image_dataset.metadata['attributes_input']['attribute:vehicle_presence'] = image_dataset.metadata['attributes_input']['attribute:vehicle_count'] > 0

## Sampling process

**Input:**
A dataset of synthetic images with adjusted attributes, unconfirmed vehicle presence sample removed, also the samples which contain unassigned attribute values after relabeling.

1. Randomly select an image from the filtered list
1. Determine which attributes to be edited:
    1. Calculate inverted attribute histograms
    1. Randomly sample new attribute values based on inverted histograms.
    1. If no attribute change is yielded skip editing and start from Step 1.
1. Construct a prompt (template or ChatGPT or local GPT)
1. Submit editing query (Gemini)
1. Perform attribute analysis of the generated image and adjust its attributes
    1. Ask Gemini to confirm the input attribute.
    1. If the answer is Yes: continue
    1. If the answer is No:
        1. Ask Gemini to predict attribute value
        1. Find the CosSim closest input attribute above a threshold (e.g. 0.57) and use it.
        1. If unable to find a match, assign the predicted Gemini Value for later or just assign N/A?

1. Update dataset attribute histograms and visualize (visdom)
1. Check for histograms uniformity: If termination criterion is reached, TERMINATE
1. Continue

In [ ]:
# Initialize the Google GenAI provider
provider_google = ag.providers.google.GoogleGenAI(api_key=config['providers']['google']['api_key'])

attrib_names = [
    'scene',
    'season',
    'weather',
    'vehicle_presence',
]

attrib_names_to_edit = [
    'season',
    'weather',
    'vehicle_presence',
]


In [ ]:
models_all = provider_google.list_models()
# models_all[models_all.display_name.apply(lambda x: x.lower().find('gemini') >= 0)]
models_all[models_all.display_name.apply(lambda x: x.lower().find('nano') >= 0)]

In [ ]:
images_metadata_filtered = image_dataset.metadata['attributes_filtered:gemini-2.5-flash'].copy()

In [ ]:
images_metadata_editing = images_metadata_filtered[
    [
        'image_file_name',
        'gen:scene_adjusted',
        'gen:season_adjusted',
        'gen:weather_adjusted',
        'gen:vehicle_presence',
    ]
].copy()
images_metadata_editing

In [ ]:
def get_attrib_col_name(attrib_name):
    if attrib_name == 'vehicle_presence':
        attrib_col_name = f'gen:{attrib_name}'
    else:
        attrib_col_name = f'gen:{attrib_name}_adjusted'
    return attrib_col_name


# Estimate minimum number of images to be edited
print(f'Estimate minimum number of images to be edited based on the initial attribute histograms...')
attrib_min_editing_counts = {}
for attrib_name in attrib_names:
    attrib_col_name = get_attrib_col_name(attrib_name)

    vc_attrib = images_metadata_editing[attrib_col_name].value_counts()
    attrib_min_editing_counts[attrib_name] = (vc_attrib.max() - vc_attrib).sum()
    print(f'   {attrib_name}: {attrib_min_editing_counts[attrib_name]}')

# int(1.05*sum(attrib_min_editing_counts.values()))

In [ ]:
# START VISDOM SERVER FIRST USING: python -m visdom.server -port 8097

vis = visdom.Visdom(server="http://localhost", port=8097)

In [ ]:
logging_folder_path = "output/logging"
os.makedirs(logging_folder_path, exist_ok=True)

# Create a custom logger
logger = logging.getLogger("image_editing")
logger.setLevel(logging.DEBUG)

# for handler in logger.handlers[:]:
#     logger.removeHandler(handler)

# Create handlers
timestamp = datetime.now().strftime('%Y-%m-%dT%H:%M:%SZ')
file_handler = logging.FileHandler(f"{logging_folder_path}/image_editing_{timestamp}.log")
console_handler = logging.StreamHandler(stream=sys.stdout)

# Set levels
file_handler.setLevel(logging.DEBUG)
console_handler.setLevel(logging.INFO)
# console_handler.setLevel(logging.DEBUG)

# Create formatter
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")

# Add formatter to handlers
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

# Add handlers to logger
logger.addHandler(console_handler)
logger.addHandler(file_handler)

In [ ]:
num_attribs_to_edit = 1
assert num_attribs_to_edit <= len(attrib_names_to_edit), "num_attribs_to_edit must be less than or equal to the number of attributes to edit"

model_name_prompt_composition   = 'gemini-2.5-flash-lite'
model_name_attribute_extraction = "gemini-2.5-flash"

model_name_image_editing = 'gemini-2.5-flash-image' # NanoBanana
# model_name_image_editing = 'gemini-2.5-flash-image-preview' # NanoBanana
# model_name_image_editing = 'gemini-2.5-flash'
# model_name_image_editing = 'gemini-2.5-pro'

model_config = ag.providers.google.types.GenerateContentConfig(
    temperature=0.0,
    thinking_config=ag.providers.google.types.ThinkingConfig(
        thinking_budget=0,
    )
)

kwargs_attribute_extraction = {
    'config': model_config,
    'is_json_response': True,
}

# Create an empty dataset to hold the edited images
image_dataset_editing = ag.image.data.Dataset(
    root_folder_path="output/TestDataset_02_Edited_Images" 
)
if len(image_dataset_editing) == 0:
    image_dataset_editing.make_dirs()

# FOLDERS
attributes_input_cache_folder_path = os.path.join(
    image_dataset_editing.cache_folder_path,
    'attributes_input',
    model_name_prompt_composition.replace('/', '_')
)
os.makedirs(attributes_input_cache_folder_path, exist_ok=True)

attributes_generated_cache_folder_path = os.path.join(
    image_dataset_editing.cache_folder_path,
    'attributes_generated',
    model_name_attribute_extraction.replace('/', '_')
)
os.makedirs(attributes_generated_cache_folder_path, exist_ok=True)


figures_output_folder_path = os.path.join(
    image_dataset_editing.cache_folder_path,
    'figures_image_attribute_editing'
)
figures_output_folder_path_histograms = os.path.join(figures_output_folder_path, 'attribute_histograms')
figures_output_folder_path_stds = os.path.join(figures_output_folder_path, 'attribute_stds')
figures_output_folder_path_image_samples = os.path.join(figures_output_folder_path, 'image_samples')
figures_output_folder_path_transition_matrices = os.path.join(figures_output_folder_path, 'transition_matrices')
os.makedirs(figures_output_folder_path_histograms, exist_ok=True)
os.makedirs(figures_output_folder_path_stds, exist_ok=True)
os.makedirs(figures_output_folder_path_image_samples, exist_ok=True)
os.makedirs(figures_output_folder_path_transition_matrices, exist_ok=True)

num_images_to_edit = int(1.75*max(attrib_min_editing_counts.values()))
logger.info(f'Number of images to be edited: {num_images_to_edit:,d}')
logger.info(f'Starting image attribute editing... (Initial Set of Images: {images_metadata_editing.shape[0]:,d})')

attrib_names_mapping = {
    'scene': 'scene_type',
    'season': 'season',
    'weather': 'weather',
    'vehicle_presence': 'vehicle_presence',
}

forbidden_attribute_combinations = [
    {'season': 'summer', 'weather': 'snowy'},
    {'season': 'spring', 'weather': 'snowy'},
    {'season': 'fall',   'weather': 'snowy'},
    {'scene':  'desert', 'weather': 'snowy'},
    {'scene':  'desert', 'season': 'spring'},
    {'scene':  'desert', 'season': 'winter'},
]

forbidden_attribute_values = {
    # 'scene': ['N/A', 'dense', 'overgrown', 'plain', 'scrubland'],
    'scene': ['N/A'],
    'season': ['N/A'],
    # 'weather': ['N/A', 'cloudless', 'dry', 'overcast'],
    'weather': ['N/A'],
}

vis_window_id_image_samples = None
vis_window_id_arrib_hists = None
vis_window_id_arrib_stds = None
vis_window_id_trans_mats = {
    'scene': None,
    'season': None,
    'weather': None,
    'vehicle_presence': None,
}
vis.close()

attribs_std = {}

# Initialize transition matrices
attrib_trans_mats_edited = {}
attrib_trans_mats_unedited = {}
for attrib_name in attrib_names:
    attrib_col_name = get_attrib_col_name(attrib_name)
    attrib_vals = sorted(images_metadata_editing[attrib_col_name].unique().tolist())

    # Create a square DataFrame with row and column names from attrib_vals
    df_square = pd.DataFrame(
        0, 
        index=attrib_vals, 
        columns=attrib_vals,
    )

    attrib_trans_mats_edited[attrib_name]   = df_square
    attrib_trans_mats_unedited[attrib_name] = df_square.copy()


# 
attribs_uniform = []
print_offset_iter = len(str(num_images_to_edit))
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
# images_metadata_attributes_input = []
images_metadata_attributes_generated = []
errors = []
logger.info(f'Starting image editing loop for {num_images_to_edit:,d} images...')
for i in tqdm(range(num_images_to_edit), desc='Image Editing'):
    sampling_success = False
    processing_sample_image = False
    processing_image_editing = False
    processing_attribute_extraction = False

    logger.debug(f'Iter.: {i+1: >{print_offset_iter},d}/{num_images_to_edit}')
    file_names_head = f"{i:0>6d}" # For naming output files

    # Make output file names
    image_metadata_input_file_path = os.path.join(
        attributes_input_cache_folder_path,
        file_names_head + '.json'
    )
    image_edited_file_name = file_names_head + '.png'
    image_edited_file_path = os.path.join(
        image_dataset_editing.images_folder_path,
        image_edited_file_name
    )
    attributes_extraction_response_file_path = os.path.join(
        attributes_generated_cache_folder_path,
        file_names_head + '.json'
    )

    # if os.path.isfile(image_metadata_input_file_path) and os.path.isfile(image_edited_file_path) and os.path.isfile(attributes_extraction_response_file_path):
    #     continue

    # print(i, os.path.isfile(image_metadata_input_file_path), os.path.isfile(image_edited_file_path), os.path.isfile(attributes_extraction_response_file_path))
    # # continue
    # assert False

    # STEP 1: Sample an image to edit
    logger.debug(f"({i+1: >{print_offset_iter},d}) STEP 1: Sample a source image to edit.")

    if os.path.isfile(image_metadata_input_file_path):
        # Load from cache
        logger.debug(f"({i+1: >{print_offset_iter},d}) STEP 1: Loading input attributes from cache ({image_metadata_input_file_path}).")
        
        image_metadata_input = pd.read_json(image_metadata_input_file_path)
        assert image_metadata_input.shape[0] == 1

        src_metadata = image_metadata_input.T.iloc[0:5]
        src_metadata.rename(index=lambda x: x.replace('src:', ''), inplace=True)
        src_metadata.rename(columns={src_metadata.columns[0]: 'Source'}, inplace=True)

        dst_metadata = image_metadata_input.T.iloc[5:]
        dst_metadata.rename(
            index={
                'attribute:scene_type': 'gen:scene_adjusted',
                'attribute:season': 'gen:season_adjusted',
                'attribute:weather': 'gen:weather_adjusted',
                'attribute:vehicle_presence': 'gen:vehicle_presence',
            },
            inplace=True)
        dst_metadata.rename(columns={dst_metadata.columns[0]: 'Destination'}, inplace=True)

        image_metadata_all = src_metadata.join(dst_metadata)
        prompt_image_editing = image_metadata_input['prompt_editing'].iloc[0]

    else:
        processing_sample_image = True
        
        num_attempts_max = 500
        for i_attempt in range(num_attempts_max):
            # Select attributes to edit
            logger.debug(f"({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1a: Select {num_attribs_to_edit} attributes to edit.")
            # attribs_to_edit = random.sample(attrib_names, k=num_attribs_to_edit)
            attribs_to_edit = random.sample(attrib_names_to_edit, k=num_attribs_to_edit)
            logger.debug(f"({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1a: Attributes selected for editing: {str(attribs_to_edit)}")

            # Sample attribute values based on their inverse histograms
            logger.debug(f"({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1b: Sample attribute values.")
            
            attrib_vals = []
            for attrib_name in attrib_names:
                attrib_col_name = get_attrib_col_name(attrib_name)

                vc_attrib = images_metadata_editing[attrib_col_name].value_counts()

                h_attrib = (1.05*vc_attrib.max()) - vc_attrib # Offsetting max bar by one to make its probability non-zero
                # h_attrib.drop(labels='N/A', inplace=True, errors='ignore')
                h_attrib.drop(labels=forbidden_attribute_values.get(attrib_name, []), inplace=True, errors='ignore')
                assert h_attrib.sum() > 0
                h_attrib_inv = h_attrib / h_attrib.sum()

                if ((vc_attrib.max() - vc_attrib == 0).all()) and (attrib_name not in attribs_uniform):
                    attribs_uniform.append(attrib_name)

                cat_attrib_vals = {
                    'attrib_name': attrib_name,
                    'to_edit': attrib_name in attribs_to_edit,
                }
                for purpose in ['src', 'dst']:
                    if (attrib_name in attribs_uniform) or (purpose == 'src' and (attrib_name in attribs_to_edit)):
                        weights = None

                    # elif purpose == 'src':
                    #     weights = h_attrib.to_list()

                    else: 
                        weights = h_attrib_inv.to_list()

                    attrib_val = random.choices(
                        h_attrib_inv.index.to_list(), 
                        weights=weights,
                        k=1
                    )[0]

                    cat_attrib_vals[f'{purpose}_val'] = attrib_val

                # image_metadata_dst[attrib_col_name] = attrib_val_edit
                # image_metadata_all.loc[attrib_col_name, 'Destination'] = attrib_val_edit
                attrib_vals.append(cat_attrib_vals)
            attrib_vals = pd.DataFrame(attrib_vals).set_index('attrib_name')
            logger.debug(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1b: Selected attribute values:\n{attrib_vals.to_dict()}')

            logger.debug(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1c: Sample images with the selected values.')
            masks = []
            for attrib_name, row in attrib_vals[~attrib_vals['to_edit']].drop(columns=['to_edit', 'dst_val']).iterrows():
                attrib_col_name = get_attrib_col_name(attrib_name)
                # masks.append(images_metadata_editing[attrib_col_name] == row['src_val'])
                masks.append(images_metadata_filtered[attrib_col_name] == row['src_val'])

            mask_to_edit = pd.concat(masks, axis=1).all(axis=1)
            # images_metadata_editing_ = images_metadata_editing[mask_to_edit]
            images_metadata_editing_ = images_metadata_filtered[mask_to_edit]
            
            if images_metadata_editing_.shape[0] == 0:
                logger.warning(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1c: No images found with the selected attribute values. Resampling...')
                continue

            image_metadata_src = images_metadata_editing_.sample().iloc[0]
            image_metadata_dst = image_metadata_src.copy()

            image_metadata_all = pd.DataFrame(image_metadata_src[['image_file_name', 'gen:scene_adjusted', 'gen:season_adjusted', 'gen:weather_adjusted', 'gen:vehicle_presence']])
            image_metadata_all.rename(columns={image_metadata_all.columns[0]: 'Source'}, inplace=True)
            image_metadata_all['Destination'] = image_metadata_all['Source']

            image_metadata_all.loc['image_file_name', 'Destination'] = image_edited_file_name

            for attrib_name, row in attrib_vals[attrib_vals['to_edit']].drop(columns=['to_edit', 'src_val']).iterrows():
                attrib_col_name = get_attrib_col_name(attrib_name)
                image_metadata_all.loc[attrib_col_name, 'Destination'] = row['dst_val']
            logger.debug(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1c: Successfull image sampling.\n{image_metadata_all.to_dict()}')
            
            # Testing
            logger.debug(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1d: Testing sampled image attributes (before and after editing ).')
            ## 1 - Ensure that at least one attribute is actually changed
            tests = []
            for attrib_name in attrib_names:
                attrib_col_name = get_attrib_col_name(attrib_name)
                tests.append(image_metadata_all.loc[attrib_col_name, 'Source'] == image_metadata_all.loc[attrib_col_name, 'Destination'])

            if all(tests):
                logger.warning(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1d: Edited image has the same attribute values as the source image. Resampling...')
                continue

            ## 2 - Ensure that no "N/A" values are in the edited attributes
            if any(map(lambda attrib_name: image_metadata_all.loc[get_attrib_col_name(attrib_name), 'Destination'] == 'N/A', attrib_names)):
                logger.warning(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1d: Edited image has "N/A" attribute(s). Resampling...')
                continue
            
            ## 3 - Ensure that no forbidden (impossible) attribute combinations are in the edited attributes
            tests = []
            for attrib_comb in forbidden_attribute_combinations:
                attrib_list = []
                attrib_col_names = []
                for attrib_name, attrib_val in attrib_comb.items():
                    attrib_col_name = get_attrib_col_name(attrib_name)
                    attrib_list.append((attrib_col_name, attrib_val))
                    attrib_col_names.append(attrib_col_name)

                attrib_list = pd.Series(dict(attrib_list))
                
                tests.append((image_metadata_all['Destination'][attrib_col_names] == attrib_list.values).all())

            if any(tests):
                logger.warning(f'({i+1: >{print_offset_iter},d}, att.: {i_attempt+1}) STEP 1d: Edited image has forbidden (impossible) attribute combination(s). Resampling...')
                continue

            sampling_success = True
            break

        if not sampling_success:
            logger.error(f'({i+1: >{print_offset_iter},d}) STEP 1: All attempts to sample an image for editing were unsuccessful. Terminate editing.')
            raise RuntimeError
            
        # END OF STEP 1 (Image Sampling)
        
        assert all(map(lambda attrib_name: image_metadata_all.loc[get_attrib_col_name(attrib_name), 'Destination'] != 'N/A', attrib_names)), "No 'N/A's allowed."

        logger.debug(f'({i+1: >{print_offset_iter},d}) From: {image_metadata_src[["gen:scene_adjusted", "gen:season_adjusted", "gen:weather_adjusted", "gen:vehicle_presence"]].to_dict()}')
        logger.debug(f'({i+1: >{print_offset_iter},d}) To:   {image_metadata_all.loc[[get_attrib_col_name(attrib_name) for attrib_name in attrib_names], "Destination"].to_dict()}')

        # STEP 2: Generate Prompt
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 2: Compose image editing prompt using "{model_name_prompt_composition}" and the set of attributes selected.')
        
        ## 1
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 2a: Start')
        
        attributes_edited = {}
        attribute_categories_unedited = {}
        for attrib_name in attrib_names:
            attrib_display_name = attrib_name.replace('_', ' ')

            if attrib_name == 'vehicle_presence':
                attrib_col_name = f'gen:{attrib_name}'
            else:
                attrib_col_name = f'gen:{attrib_name}_adjusted'

            if image_metadata_all["Source"][attrib_col_name] == image_metadata_all["Destination"][attrib_col_name]:
                attribute_categories_unedited[attrib_display_name] = image_metadata_all.loc[attrib_col_name, 'Source']
            else:
                # attributes_edited[attrib_display_name] = image_metadata_all["Destination"][attrib_col_name]
                attributes_edited[attrib_display_name] = tuple(image_metadata_all.loc[attrib_col_name].to_list())

            logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 2a: {attrib_display_name}: {image_metadata_all["Source"][attrib_col_name]} -> {image_metadata_all["Destination"][attrib_col_name]}')

        ##
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 2b: Construct a query template-based prompt.')
        prompt_composition_request = "Compose a prompt for editing an aerial top-down view image. The following image attribute have to be edited: "
        for attrib_name, vals in attributes_edited.items():
            if attrib_name != 'vehicle presence':
                if vals[0] == 'N/A':
                    p = f'the {attrib_name} should be set to {vals[1]}'
                else:
                    p = f'the {attrib_name} should be changed from {vals[0]} to {vals[1]}'
            else:
                if vals[1]:
                    p = f'vehicles should be added to the image'
                else:
                    p = f'all existing vehicles in the image should be removed'
            prompt_composition_request += p + "; "

        if len(attribute_categories_unedited) > 0:
            prompt_composition_request += "The following image attributes should remain unchanged: "
            for attrib_name, attrib_value in attribute_categories_unedited.items():
                if attrib_value != 'N/A':
                    p = f'the {attrib_name} should remain as {attrib_value}'
                else:
                    p = f'the {attrib_name} should remain unchanged'
                prompt_composition_request += p + "; "

        # prompt_composition_request += "The prompt should be concise and clear. "
        prompt_composition_request += "The edited image should also be an aerial top-down view image. "
        prompt_composition_request += "Return only the prompt without any additional text."
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 2b: Constructed prompt: "{prompt_composition_request}"')
        
        ##
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 2c: Submit the query...')

        try:
            response = provider_google.client.models.generate_content(
                model=model_name_prompt_composition,
                contents=prompt_composition_request,
            )
            prompt_image_editing = response.text.strip().strip('"').strip("'")
            logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 2c: Generated image editing prompt: {prompt_image_editing}')

        except Exception as e:
            logger.warning(f"({i+1: >{print_offset_iter},d}) STEP 2c: Error during prompt composition: {e} | Skipping image editing @ {i} iteration.")
            errors.append(f'[Iter: {i}] Prompt composition failure: {e}')
            continue

        
        ## Store Input Attributes Metadata
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 3: Save input attributes metadata.')
        image_metadata_src = image_metadata_all['Source'].copy()
        image_metadata_src.rename(
            index={
                'image_file_name':      'src:image_file_name',
                'gen:scene_adjusted':   'src:gen:scene_adjusted',
                'gen:season_adjusted':  'src:gen:season_adjusted',
                'gen:weather_adjusted': 'src:gen:weather_adjusted',
                'gen:vehicle_presence': 'src:gen:vehicle_presence'
            }, 
            inplace=True
        )

        image_metadata_dst = image_metadata_all['Destination'].copy()
        image_metadata_dst.rename(
            index={
                # 'image_file_name':      'image_file_name',
                'gen:scene_adjusted':   'attribute:scene_type',
                'gen:season_adjusted':  'attribute:season',
                'gen:weather_adjusted': 'attribute:weather',
                'gen:vehicle_presence': 'attribute:vehicle_presence'
            }, 
            inplace=True
        )
        image_metadata_dst['prompt_editing'] = prompt_image_editing

        image_metadata_input = pd.concat([image_metadata_src, image_metadata_dst]).to_frame().T.iloc[0]
        # images_metadata_attributes_input.append(image_metadata_input)
        image_metadata_input.to_json(image_metadata_input_file_path)
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 3: Input attributes metadata saved to: {image_metadata_input_file_path}')
   
    
    # EDIT IMAGE
    logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 4: Edit the input image using model "{model_name_image_editing}".')
    
    image_src_data = image_dataset[image_metadata_all.loc['image_file_name', 'Source']]
    image_src = image_src_data['image']

    if os.path.isfile(image_edited_file_path) and not processing_sample_image:
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 4: Edited image already exists ({image_edited_file_path}). Skipping image editing and loading from file ({image_edited_file_path}).')
        image_edited = Image.open(image_edited_file_path)

    else:
        processing_image_editing = True

        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 4: Editing image: {image_metadata_all.loc["image_file_name", "Source"]} -> {image_metadata_all.loc["image_file_name", "Destination"]}')
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 4: With prompt:\n{prompt_image_editing}')
        try:
            response = provider_google.client.models.generate_content(
                model=model_name_image_editing,
                contents=[prompt_image_editing, image_src],
            )
            logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 4: Image editing finished.')

            for part in response.candidates[0].content.parts:
                if part.text is not None:
                    logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 4: Model text response: "{part.text}"')
                    
                elif part.inline_data is not None:
                    image_edited = Image.open(BytesIO(part.inline_data.data))
                    image_edited.save(image_edited_file_path)

                    logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 4: Edited image saved to: {image_edited_file_path}')

        except Exception as e:
            logger.warning(f"({i+1: >{print_offset_iter},d}) STEP 4: Error during image editing: {e} | Skipping image editing @ {i} iteration.")
            errors.append(f'[Iter: {i}] Image editing failure: {e}')
            continue


    # CONFIRM EDITED IMAGE ATTRIBUTES
    logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 5: Extract edited image attributes.')
    
    if os.path.isfile(attributes_extraction_response_file_path) and not processing_sample_image and not processing_image_editing:
        # Load from cache
        logger.debug(f"({i+1: >{print_offset_iter},d}) STEP 5: Loading attribute extraction response from cache ({attributes_extraction_response_file_path}).")
        
        with open(attributes_extraction_response_file_path, "r") as f:
            response = json.load(f)
            
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 5: Extracted attributes:\n{response["response_parsed"]}')
        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 5: Attribute extraction response loaded from: {attributes_extraction_response_file_path}')

    else:
        processing_attribute_extraction = True

        scene_vals = sorted(set(images_metadata_editing['gen:scene_adjusted'].unique()).difference(set(forbidden_attribute_values['scene'])))
        weather_vals = sorted(set(images_metadata_editing['gen:weather_adjusted'].unique()).difference(set(forbidden_attribute_values['weather'])))
        
        prompt = textwrap.dedent(
            f"""Extract the following attributes from the image and return them as a JSON object. If an attribute cannot be determined from the image, set its value to "N/A". Answer with a single expression per attribute.

            {{
            "camera_view_confirmation": "The answer of the following question: Is the camera view depicted in this image a top-down vertical (a.k.a. nadir) view? Answer with yes or no.",
            "scene_type": "A string describing the environment in this aerial top-down view image, selected from the following list {str(scene_vals)} or N/A if unable to determine.",
            "scene_concepts": ["a list of concepts describing the scene in the image"],
            "scene_type_confirmation": "The answer of the following question: Can the environment depicted in this image be described as a {image_metadata_all['Destination']['gen:scene_adjusted']} environment? Answer with yes or no.",
            "season": "A string indicating the season, selected from the following list ['spring', 'summer', 'fall', 'winter'] or N/A if unable to determine.",
            "season_confirmation": "The answer of the following question: Can the season depicted in this image be described as {image_metadata_all['Destination']['gen:season_adjusted']}? Answer with yes or no.",
            "weather": "A string describing the weather conditions from the following list {str(sorted(images_metadata_editing[images_metadata_editing['gen:weather_adjusted'] != 'N/A']['gen:weather_adjusted'].unique()))} or N/A if unable to determine.",
            "weather_confirmation": "The answer of the following question: Can the weather conditions depicted in this image be described as {image_metadata_all['Destination']['gen:weather_adjusted']}? Answer with yes or no.",
            "vehicle_presence": "The answer of the following question: Are there any vehicles in this image? The answer should be binary (True or False).",
            }}

            Example:
            Input: 
            Output: {{
            "camera_view_confirmation": "yes",
            "scene_type": "urban",
            "scene_concepts": ["urban environment", "trees", "parking lot"],
            "scene_type_confirmation": "yes",
            "season": "autumn",
            "season_confirmation": "yes",
            "weather": "overcast",
            "weather_confirmation": "yes",
            "vehicle_presence": true
            }}"""
        )

        logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 5: Prompt\n"{prompt}"')

        try:
            response = provider_google.query_image_prompt(
                image=image_edited_file_path,
                prompt=prompt,
                model_name=model_name_attribute_extraction,
                **kwargs_attribute_extraction
            )
            
            assert response.get('response_parsed', None) is not None, "No parsed response found."

            with open(attributes_extraction_response_file_path, "w") as f:
                json.dump(response, f)
                
            logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 5: Extracted attributes:\n{response["response_parsed"]}')
            logger.debug(f'({i+1: >{print_offset_iter},d}) STEP 5: Attribute extraction response saved to: {attributes_extraction_response_file_path}')

        except Exception as e:
            logger.warning(f"({i+1: >{print_offset_iter},d}) STEP 5: Error during attribute extraction: {e} | Skipping image editing @ {i} iteration.")
            errors.append(f'[Iter: {i}] Attribute extraction failure: {e}')
            continue


    # Append the edited image metadata to the editing DataFrame
    image_metadata_adjusted = pd.Series(
        {
            "image_file_name": image_metadata_all['Destination']['image_file_name'],
            "gen:scene_adjusted": response['response_parsed']['scene_type'],
            "gen:season_adjusted": response['response_parsed']['season'],
            "gen:weather_adjusted": response['response_parsed']['weather'],
            "gen:vehicle_presence": response['response_parsed']['vehicle_presence'],
        }
    )
    
    images_metadata_editing = pd.concat(
        [images_metadata_editing, image_metadata_adjusted.to_frame().T],
        ignore_index=True
    )

    # LOG STD OF ATTRIBUTE HISTOGRAMS FOR PLOTTING
    for attrib_name in attrib_names:
        attrib_col_name = get_attrib_col_name(attrib_name)
        vc_attrib_editing = images_metadata_editing[attrib_col_name].value_counts(sort=False).sort_index()
        attrib_vals_forbidden = forbidden_attribute_values.get(attrib_name, [])
        attribs_std.setdefault(attrib_name, []).append(vc_attrib_editing[[v not in attrib_vals_forbidden for v in vc_attrib_editing.index]].std())

    # UPDATE TRANSITION MATRICES
    attrib_col_names = image_metadata_all.index[1:]
    mask_edited = image_metadata_all.loc[attrib_col_names, 'Source'] != image_metadata_all.loc[attrib_col_names, 'Destination']

    ## Edited
    for attrib_col_name in attrib_col_names[mask_edited]:
        col_name_split = attrib_col_name.split(':')[1].split('_')
        if col_name_split[-1] == 'adjusted':
            attrib_name = '_'.join(col_name_split[:-1])
        else:
            attrib_name = '_'.join(col_name_split)

        attrib_trans_mats_edited[attrib_name].loc[image_metadata_all.loc[attrib_col_name, 'Destination'], response['response_parsed'][attrib_names_mapping[attrib_name]]] += 1

    ## Unedited
    for attrib_col_name in attrib_col_names[~mask_edited]:
        col_name_split = attrib_col_name.split(':')[1].split('_')
        if col_name_split[-1] == 'adjusted':
            attrib_name = '_'.join(col_name_split[:-1])
        else:
            attrib_name = '_'.join(col_name_split)
            
        attrib_trans_mats_unedited[attrib_name].loc[image_metadata_all.loc[attrib_col_name, 'Destination'], response['response_parsed'][attrib_names_mapping[attrib_name]]] += 1


    if any([processing_sample_image, processing_image_editing, processing_attribute_extraction]): # Skip plotting if loading from cache
        # VISTOM REPORTING
        ## IMAGE SAMPLES
        attributes_edited = {}
        for attrib_name in attrib_names:
            attrib_display_name = attrib_name.replace('_', ' ')

            if attrib_name == 'vehicle_presence':
                attrib_col_name = f'gen:{attrib_name}'
            else:
                attrib_col_name = f'gen:{attrib_name}_adjusted'

            if image_metadata_all["Source"][attrib_col_name] == image_metadata_all["Destination"][attrib_col_name]:
                continue
                
            attributes_edited[attrib_display_name] = tuple(image_metadata_all.loc[attrib_col_name].to_list())

        fig, axs = plt.subplots(1, 2, figsize=(15, 7.5))
        axs[0].imshow(image_src)
        axs[0].set_title("Original Image")
        axs[0].axis('off')

        axs[1].imshow(image_edited)
        axs[1].set_title("Edited Image")
        axs[1].axis('off')
        fig.suptitle(f'Image Editing Iteration {i+1: >{print_offset_iter},d} | {";; ".join([f"{k}: {v[0]} -> {v[1]}" for k,v in attributes_edited.items()])}')
        fig.tight_layout()
        fig.savefig(f'{figures_output_folder_path_image_samples}/{i+1:06d}.png', dpi=150)
        # fig.savefig(f'{figures_output_folder_path_image_samples}/{i+1:06d}.jpg', dpi=150)

        if vis_window_id_image_samples is None:
            vis_window_id_image_samples = vis.matplot(fig)
        else:
            vis.matplot(fig, win=vis_window_id_image_samples)

        plt.close(fig)

        
        ## PLOT ATTRIBUTE HISTOGRAMS
        fig, axs = plt.subplots(
            # nrows=1, ncols=len(attrib_names),
            # figsize=(22, 5),
            nrows=2, ncols=len(attrib_names)//2,
            figsize=(11, 9),
            # dpi=150
        )
        axs = axs.flatten()

        for attrib_name, ax in zip(attrib_names, axs):
            attrib_col_name = get_attrib_col_name(attrib_name)
            attrib_display_name = ' '.join(list(map(lambda x: x.capitalize(), attrib_name.split('_'))))
            
            plot_title = f'Attribute Histogram: {attrib_display_name}'
            if attrib_name in attribs_uniform: # TODO: This is not accurate if the cache is used
                plot_title += ' (Uniform Mode)'

            vc_attrib_editing  = images_metadata_editing[attrib_col_name].value_counts(sort=False).sort_index()
            vc_attrib_editing.plot(
                kind='bar', 
                title=plot_title, 
                ax=ax, color=sns.color_palette()[1], alpha=1
            )

            # attrib_vals_forbidden = forbidden_attribute_values.get(attrib_name, [])
            # attribs_std.setdefault(attrib_name, []).append(vc_attrib_editing[[v not in attrib_vals_forbidden for v in vc_attrib_editing.index]].std())

            vc_attrib  = images_metadata_filtered[attrib_col_name].value_counts(sort=False).sort_index()
            vc_attrib.plot(
                kind='bar', 
                title=plot_title, 
                ax=ax, color=sns.color_palette()[0], alpha=1
            )
            ax.set_xlabel(None)
            ax.set_ylabel("Number of Images")
            ax.legend(['Edited Images', 'Generated Images'], loc='lower center', ncol=2)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

        fig.suptitle(f'Image Attribute Histograms After Editing Iteration {i+1}')
        fig.tight_layout()
        fig.savefig(f'{figures_output_folder_path_histograms}/{i+1:06d}.png', dpi=150)
        # fig.savefig(f'{figures_output_folder_path_histograms}/{i+1:06d}.jpg', dpi=150)

        if vis_window_id_arrib_hists is None:
            vis_window_id_arrib_hists = vis.matplot(fig)
        else:
            vis.matplot(fig, win=vis_window_id_arrib_hists)

        plt.close(fig)


        ## PLOT ATTRIBUTE STDS
        fig, axs = plt.subplots(
            nrows=2, ncols=len(attrib_names)//2,
            figsize=(11, 9)
        )
        axs = axs.flatten()
        for attrib_name, stds in attribs_std.items():
            ax = axs[attrib_names.index(attrib_name)]
            ax.plot(
                range(1, len(stds)+1), 
                stds, 
                # marker='o'
            )
            ax.set_title(f'Std. Dev. of { " ".join(attrib_name.split("_")).capitalize() }')
            ax.set_xlabel('Iterations')
            ax.set_ylabel('')
            # ax.set_xticks(range(1, len(stds)+1))
            ax.grid(True)

        fig.tight_layout()
        fig.savefig(f'{figures_output_folder_path_stds}/{attrib_name}_std.png', dpi=150)

        if vis_window_id_arrib_stds is None:
            vis_window_id_arrib_stds = vis.matplot(fig)
        else:
            vis.matplot(fig, win=vis_window_id_arrib_stds)

        plt.close(fig)
            

        ## PLOT TRANSITION MATRICES
        for attrib_name in attrib_names:
            fig_size = 6 + attrib_trans_mats_edited[attrib_name].shape[0]//2
            fig, axs = plt.subplots(1, 2, figsize=(fig_size, fig_size//2))

            sns.heatmap(
                attrib_trans_mats_edited[attrib_name],
                annot=True, fmt='d',
                cmap='Blues',
                cbar=False,
                square=True,
                linewidths=0.5,
                linecolor='gray',
                annot_kws={"size": 6},
                ax=axs[0]
            )
            axs[0].set_xlabel('Edited To')
            axs[0].set_ylabel('From')
            axs[0].set_title('Edited Images')

            sns.heatmap(
                attrib_trans_mats_unedited[attrib_name],
                annot=True, fmt='d',
                cmap='Blues',
                cbar=False,
                square=True,
                linewidths=0.5,
                linecolor='gray',
                annot_kws={"size": 6},
                ax=axs[1]
            )
            axs[1].set_xlabel('Edited To')
            axs[1].set_ylabel('From')
            axs[1].set_title('Unedited Images')

            fig.suptitle(f'Attribute Transition Matrix - {" ".join(attrib_name.split("_")).capitalize()}')
            fig.tight_layout()
            fig.savefig(f'{figures_output_folder_path_transition_matrices}/{i+1:06d}_{attrib_name}.png', dpi=150)

            if vis_window_id_trans_mats[attrib_name] is None:
                vis_window_id_trans_mats[attrib_name] = vis.matplot(fig)
            else:
                vis.matplot(fig, win=vis_window_id_trans_mats[attrib_name])

            plt.close(fig)

    # break

logger.info(f'Finished image attribute editing.')

# Process Image Attributes

In [ ]:
attributes_input_cache_folder_path = os.path.join(image_dataset_editing.cache_folder_path, 'attributes_input', 'gemini-2.5-flash-lite')
attributes_generated_cache_folder_path = os.path.join(image_dataset_editing.cache_folder_path, 'attributes_generated', 'gemini-2.5-flash')


## Input Attributes Metadata

In [ ]:
# Load all input attributes metadata
metadata_attributes_input = []
for fn in sorted(os.listdir(attributes_input_cache_folder_path)):
    fp = os.path.join(attributes_input_cache_folder_path, fn)

    image_attributes_input_metadata = pd.read_json(fp, typ='series')
    metadata_attributes_input.append(image_attributes_input_metadata)

metadata_attributes_input = pd.DataFrame(metadata_attributes_input)
metadata_attributes_input.rename(columns={'prompt_editing': 'prompt_editing:gemini-2.5-flash-lite'}, inplace=True)
image_dataset_editing.metadata['attributes_input'] = metadata_attributes_input
metadata_attributes_input

## Generated Attributes Metadata

In [ ]:
metadata_attributes_generated = []
for fn in sorted(os.listdir(attributes_generated_cache_folder_path)):
    fp = os.path.join(attributes_generated_cache_folder_path, fn)

    with open(fp, 'r') as f:
        response = json.load(f)
    metadata_attributes_generated.append(response)

metadata_attributes_generated = pd.DataFrame(metadata_attributes_generated)
metadata_attributes_generated

In [ ]:
attributes_generated = pd.DataFrame(metadata_attributes_generated['response_parsed'].to_list())
attributes_generated.rename(columns=lambda x: f"attribute:{x}", inplace=True)
attributes_generated['attribute:camera_view_confirmation'] = attributes_generated['attribute:camera_view_confirmation'].apply(lambda x: True if x.lower() == 'yes' else False)
# attributes_generated

# Combine input and generated attributes metadata
metadata_attributes_generated = metadata_attributes_generated.merge(
    attributes_generated,
    left_index=True,
    right_index=True,
    suffixes=('_input', '_generated')
)
metadata_attributes_generated

In [ ]:
image_dataset_editing.metadata['attributes_generated:gemini-2.5-flash'] = metadata_attributes_generated
image_dataset_editing.metadata['attributes_generated:gemini-2.5-flash']

In [ ]:
image_dataset_editing.save_metadata()